In [1]:
from pathlib import Path
import joblib
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.net(x)

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Test samples: {len(X_test)}")

Test samples: 30


In [3]:
rf = joblib.load("models/iris_rf.pkl")
rf_preds = rf.predict(X_test)
print("=" * 40)
print("Random Forest")
print("=" * 40)
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds, target_names=iris.target_names))

Random Forest
Accuracy: 1.0000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [4]:
cat = CatBoostClassifier()
cat.load_model("models/iris_catboost.cbm")
cat_preds = cat.predict(X_test)
print("=" * 40)
print("CatBoost")
print("=" * 40)
print(f"Accuracy: {accuracy_score(y_test, cat_preds):.4f}")
print(classification_report(y_test, cat_preds, target_names=iris.target_names))

CatBoost
Accuracy: 1.0000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [5]:
model = IrisNet()
model.load_state_dict(torch.load("models/iris_nn.pt", weights_only=True))
model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    nn_logits = model(X_test_t)
    nn_preds = nn_logits.argmax(dim=1).numpy()

print("=" * 40)
print("IrisNet (PyTorch)")
print("=" * 40)
print(f"Accuracy: {accuracy_score(y_test, nn_preds):.4f}")
print(classification_report(y_test, nn_preds, target_names=iris.target_names))

IrisNet (PyTorch)
Accuracy: 1.0000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [6]:
print("Confusion matrices:\n")
print("Random Forest:")
print(confusion_matrix(y_test, rf_preds))
print()
print("CatBoost:")
print(confusion_matrix(y_test, cat_preds))
print()
print("IrisNet:")
print(confusion_matrix(y_test, nn_preds))

Confusion matrices:

Random Forest:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

CatBoost:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

IrisNet:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
